## Aproximación por serie de Taylor de $f(x) = x^{-3}(\sin x - x)$

### Derivación

Partiendo de la serie de Taylor del seno:
$\sin x = \sum_{i=0}^{\infty} \frac{(-1)^i\, x^{2i+1}}{(2i+1)!}$

Se tiene que:
$\sin x - x = \sum_{i=1}^{\infty} \frac{(-1)^i\, x^{2i+1}}{(2i+1)!}$

Dividiendo por $x^3$ (con $x \neq 0$) y reindexando con $j = i - 1$:

$f(x) = x^{-3}(\sin x - x) = \sum_{j=0}^{\infty} \frac{(-1)^{j+1}\, x^{2j}}{(2j+3)!} = -\frac{1}{3!} + \frac{x^2}{5!} - \frac{x^4}{7!} + \cdots$

Además podemos evitar el caso en que $x = 0$ eliminando analíticamente ese término del denominador. De esta forma la serie converge para todo $x$ y es bien comportada cerca del origen.

### Recurrencia entre términos consecutivos

Siendo $t_j = \dfrac{(-1)^{j+1}\, x^{2j}}{(2j+3)!}$:

$t_{j+1} = t_j \cdot \frac{-x^2}{(2j+5)(2j+4)}$

Con `denom` $= 2j + 5$ (arranca en 5, incrementa de 2 en 2):

$t_{\text{new}} = t_{\text{prev}} \cdot \frac{-x^2}{\texttt{denom} \cdot (\texttt{denom} - 1)}$

Esto evita calcular potencias y factoriales desde cero en cada paso.

### Implementación — con precisión configurable

La función `f_aprox(x, tol, precision)` acepta un string de precisión:
- `"Float16"` → 16 bits (~3 dígitos decimales, épsilon de máquina ≈ 4.88e-4)
- `"Float32"` → 32 bits (~7 dígitos decimales, épsilon de máquina ≈ 1.19e-7)
- `"Float64"` → 64 bits (~15 dígitos decimales, épsilon de máquina ≈ 2.22e-16)

**Todos** los cálculos internos (constantes, acumuladores, operaciones) se realizan
en el tipo indicado, de modo que los errores de redondeo y representación son
exactamente los propios de esa aritmética.

In [ ]:
"""
Aproxima f(x) = x⁻³·(sin(x) - x) mediante la serie alternada de Taylor:
f(x)= Σ_{j=0}^{∞}  (-1)^{j+1} · x^{2j} / (2j+3)!
    = -1/3! + x²/5! - x⁴/7! + ...

usamos recurrencia entre términos consecutivos (usamos los terminos anteriormente calculados).
Nota: `precision` es un string con el tipo flotante a usar: "Float32" o "Float64"
"""
# Todos los cálculos internos se realizan en el tipo especificado, por lo que
# los errores de redondeo y representación son los propios de esa aritmética.
function f_aprox(a, tol, precision="Float64", max_iterations = 1000)

    # Seleccionamos el tipo numérico según el string que indica la precisión
    T = if precision == "Float32"
            Float32
        elseif precision == "Float64"
            Float64
        else
            error("Precisión no reconocida: '$precision'. Usa \"Float32\" o \"Float64\".")
        end

    # Convertimos los grados a radianes en la precisión parametrizada (T)
    x = T(a) / (T(180) / T(pi))

    # Convertimos las constantes y la entrada a la presición parametrizada (T)
    x_cuad = x ^ T(2)
    # primer término t_0 = -1/3! = -1/6  en precisión T 
    term = T(-1) / T(6)
    # res es el lugar donde iremos "acumulando" los términos 
    res = term
    # primer factor del denominador recurrente
    denom = T(5)
    # Expresamos la tolerancia en la misma presición
    tol_T = T(tol)

    
    converged = false
    count_iterations = 0

    for _ in 1:max_iterations
        # t_{j+1} = t_j * (−x^2) / [(denom)·(denom−1)]
        term = (-(term * x_cuad)) / (denom * (denom - T(1)))
        res  = res + term
        
        count_iterations += 1

        if abs(term) < tol_T
            converged = true
            break
        end
        
        denom = denom + T(2)
    end

    if !converged
        println("[WARNING]: Terminó por máximo de iteraciones ($max_iterations)")
    else
        println("[INFO]: Convergió en $count_iterations iteraciones")
    end

    return res, count_iterations
end


### Tabla comparativa de resultados por precisión

Se usa [`PrettyTables.jl`](https://github.com/ronisbr/PrettyTables.jl) para mostrar en una tabla con los valores de referencia y los errores absolutos de cada precisión para los ángulos de interés.

In [ ]:
# Valor de referencia en BigFloat — recibe grados, convierte a radianes
function f_ref(a)
    xb = BigFloat(a) / (BigFloat(180) / BigFloat(pi))
    (sin(xb) - xb) / xb^3
end

In [ ]:
import Pkg; 
Pkg.add("PrettyTables")
Pkg.status("PrettyTables")


In [ ]:
using Printf
using PrettyTables

angulos = [30, 390, 750, 1110, 1470, 1830, 2190, 2550]
tol     = 1e-6

# Calcular valores e iteraciones necesitadas
f32_val_iter = []
f64_val_iter = []

refs   = [Float64(f_ref(a)) for a in angulos]

for a in angulos    
    appr, iters = f_aprox(a, tol, "Float32")
    push!(f32_val_iter, (Float64(appr),iters))
    
    appr, iters = f_aprox(a, tol, "Float64")
    push!(f64_val_iter, (Float64(appr),iters))
end

# Obtenemos errores absolutos de cada aproximación
err32 = [abs(x[1] - r) for (x, r) in zip(f32_val_iter, refs)]
err64 = [abs(x[1] - r) for (x, r) in zip(f64_val_iter, refs)]

# Armar la tabla
data = hcat(angulos, refs, first.(f32_val_iter), first.(f64_val_iter), err32, err64)

pretty_table(
    data,
    backend       = :html,
    column_labels = [
        "Ángulo (°)", "Referencia (BigFloat)", "Float32", "Float64", "Err abs. F32", "Err abs. F64"
    ],
    formatters = [(v, i, j) -> j in 2:8 ? @sprintf("%.6e", v) : v],
    alignment  = [:r, :r, :r, :r, :r, :r],
)


### Gráfico para contrastar errores absolutos
Cada grupo de barras corresponde a un valor de entrada; dentro del grupo se comparan los errores absolutos de las tres precisiones.

In [ ]:
import Pkg; Pkg.add("Plots")
Pkg.status("Plots")


In [ ]:
using Plots
using Plots.PlotMeasures

gr()

plot(angulos, err32;
    label         = "Float32",
    color         = :steelblue,
    lw            = 2,
    marker        = :circle,
    markersize    = 6,
    yscale        = :log10,
    yticks        = [ 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1e0, 1e2, 1e4, 1e6],
    xlabel        = "Ángulo de entrada (°)",
    ylabel        = "Error absoluto (escala log₁₀)",
    title         = "\nEvolución del error absoluto por precisión",
    legend        = :topleft,
    framestyle    = :box,
    grid          = true,
    gridalpha     = 0.3,
    gridlinewidth = 0.8,
    gridcolor     = :gray,
    size = (700, 340),
    xticks        = angulos,
    xrotation     = 45,
    bottom_margin = 7mm,
    left_margin   = 7mm,
    right_margin = 5mm,
    top_margin = 7mm,

)

plot!(angulos, err64;
    label      = "Float64",
    color      = :tomato,
    lw         = 2,
    marker     = :square,
    markersize = 6,
)

### Gráfico para visualizar errores relativos por ángulo
El error relativo normaliza el error absoluto respecto al valor de referencia, permitiendo comparar la precisión independientemente de la magnitud de `f(x)`.

In [ ]:
err32_rel = err32 ./ abs.(refs)
err64_rel = err64 ./ abs.(refs)

plot(angulos, err32_rel;
    label         = "Float32",
    color         = :steelblue,
    lw            = 2,
    marker        = :circle,
    markersize    = 6,
    yscale        = :log10,
    yticks        = [1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1e0, 1e2, 1e4, 1e6, 1e8, 1e10],
    xlabel        = "Ángulo de entrada (°)",
    ylabel        = "Error relativo",
    title         = "\nEvolución del error relativo por precisión",
    legend        = :topleft,
    framestyle    = :box,
    grid          = true,
    gridalpha     = 0.3,
    gridlinewidth = 0.8,
    gridcolor     = :gray,
    size = (700, 340),
    xticks        = angulos,
    xrotation     = 45,
    bottom_margin = 7mm,
    left_margin   = 7mm,
    right_margin = 5mm,
    top_margin = 7mm,
)

plot!(angulos, err64_rel;
    label      = "Float64",
    color      = :tomato,
    lw         = 2,
    marker     = :square,
    markersize = 6,
)